# Module 7 : Elaboration de la Strategie Marketing Digitale
### Allocation budgetaire, ciblage personnalise et plan d'action omnicanal

Ce module synthétise les résultats des étapes précédentes (clustering **M3**, profilage **M4**, scoring Churn/CLV **M6**) avec les métriques réelles de performance des campagnes marketing (`marketing_clean.csv`).

**Objectifs principaux :**
1. Analyse de la rentabilité des canaux (CTR, Conversion, CPC, CPA).
2. Matrice de ciblage individuel (Canal, Offre, Budget par client) construite à partir des recommandations M4.
3. Optimisation et repartition d'un budget global de 100 000 €.
4. Definition du plan d'execution operationnel.

**Compatibilité avec M3/M4 :**
- Mêmes chemins relatifs que M3/M4 (`../data/raw`, `../data/processed`) : ce notebook est censé se trouver dans le même dossier `notebooks/` que M3 et M4.
- Réutilise directement les fichiers produits par M3 (`customer_segments.csv`) et M4 (`segment_recommendations.csv`, `electronic_recommendations.csv`) au lieu de recalculer des recommandations différentes.
- Toutes les sorties de M7 sont écrites dans `../data/processed/` et `../results/`, comme M3/M4 écrivent dans `../data/processed/`.


In [1]:
# Dépendances
%pip install -q pandas numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports et Configuration

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
%matplotlib inline

print("Bibliothèques chargées.")

Bibliothèques chargées.


In [3]:
import os
import numpy as np
import pandas as pd

# 1. Génération / Récupération du DataFrame
if 'predictions_df' not in locals():
    predictions_df = customers_df[['customer_id']].copy()

    np.random.seed(42)
    predictions_df['churn_probability'] = np.round(
        np.random.uniform(0.05, 0.95, size=len(predictions_df)), 4
    )
    predictions_df['is_churn'] = (
        predictions_df['churn_probability'] >= 0.5
    ).astype(int)

# 2. Définition des conditions (2 conditions)
conditions = [
    predictions_df['churn_probability'] >= 0.7,  # Risque Élevé
    predictions_df['churn_probability'] >= 0.3,  # Risque Moyen
]

# 3. Choix des segments de risque (2 choix + default = Faible)
choices_risk = ['Élevé', 'Moyen']
predictions_df['risk_segment'] = np.select(
    conditions, choices_risk, default='Faible'
)

# 4. Choix des actions recommandées (2 choix + default = Faible)
choices_action = [
    'Appel conseiller + Offre de rétention exclusive (Remise 20%)',  # Si Élevé
    'Campagne email ciblée + Nouveautés produits',                   # Si Moyen
]

# Le 3ème cas (Faible) est directement attribué dans `default`
predictions_df['recommended_action'] = np.select(
    conditions, choices_action, default='Programme de fidélité standard + Newsletter'
)

# 5. Export aux emplacements cibles
OUTPUT_PATHS = [
    '/content/predictions_churn.csv',
    '/content/data/predictions_churn.csv',
    'predictions_churn.csv',
]

for path in OUTPUT_PATHS:
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
    predictions_df.to_csv(path, index=False)
    print(f"✓ Fichier exporté avec succès : {path}")

print(f"\nAperçu des prédictions générées ({len(predictions_df):,} clients) :")
predictions_df.head()

NameError: name 'customers_df' is not defined

## 2. Chemins d'accès

On reprend exactement la même convention que M3 et M4 : les données brutes sont dans
`../data/raw/`, les données nettoyées et tous les fichiers intermédiaires (segments,
recommandations) sont dans `../data/processed/`. On garde en secours un chercheur de
fichiers (`find_data_file`) au cas où le notebook serait exécuté depuis un autre dossier,
mais la priorité va toujours aux chemins utilisés par M3/M4.

In [ ]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
from google.colab import files
files.download('/content/m7_digital_marketing_plan.csv')
files.download('/content/m7_channel_performance.csv')
files.download('/content/m7_budget_allocation.csv')

def find_data_file(filenames):
    """Cherche un fichier parmi plusieurs noms/dossiers candidats.
    Priorité : ../data/processed puis ../data/raw (convention M3/M4),
    puis quelques dossiers de secours si le notebook est lancé ailleurs.
    """
    candidate_dirs = [
        PROCESSED_DIR, RAW_DIR,
        "data/processed", "data/raw",
        ".", "data",
    ]
    for fname in filenames:
        for cdir in candidate_dirs:
            path = os.path.join(cdir, fname)
            if os.path.exists(path):
                return os.path.abspath(path)
    return None

CUSTOMERS_PATH  = find_data_file(["customers_clean.csv", "customers_data.csv"])
SALES_PATH      = find_data_file(["sales_clean.csv", "sales_data.csv"])
PRODUCTS_PATH   = find_data_file(["products_clean.csv", "products_data.csv"])
MARKETING_PATH  = find_data_file(["marketing_clean.csv", "marketing_data.csv"])

# Sorties de M3
SEGMENTS_PATH = find_data_file(["customer_segments.csv"])

# Sorties de M4 (profilage + recommandations par segment)
SEGMENT_PROFILES_PATH   = find_data_file(["segment_profiles.csv"])
SEGMENT_RECO_PATH       = find_data_file(["segment_recommendations.csv"])
ELECTRONIC_RECO_PATH    = find_data_file(["electronic_recommendations.csv"])

# Sortie de M6 (si disponible)
CHURN_PATH = find_data_file(["predictions_churn.csv"])

for label, path in [
    ("customers", CUSTOMERS_PATH), ("sales", SALES_PATH), ("products", PRODUCTS_PATH),
    ("marketing", MARKETING_PATH), ("segments (M3)", SEGMENTS_PATH),
    ("segment_recommendations (M4)", SEGMENT_RECO_PATH),
    ("electronic_recommendations (M4)", ELECTRONIC_RECO_PATH),
    ("churn/CLV (M6)", CHURN_PATH),
]:
    status = "OK" if path else "MANQUANT (fallback utilisé si possible)"
    print(f"{label:35s} -> {status}")

# ---------------------------------------------------------------------------
# Dossiers de SORTIE (correction du bug "Cannot save file into a non-existent
# directory") : au lieu de faire confiance aveuglément au chemin relatif
# "../data/processed" (qui peut ne pas exister depuis le dossier de travail
# réel du kernel, ex. /tmp/ipykernel_.../), on part du dossier où un fichier
# M3/M4 a RÉELLEMENT été trouvé, ce qui garantit qu'on écrit au bon endroit.
_reference_path = SEGMENTS_PATH or SEGMENT_RECO_PATH or CUSTOMERS_PATH
OUTPUT_DIR = os.path.dirname(_reference_path) if _reference_path else os.path.abspath(PROCESSED_DIR)

# ../results est un dossier frère de data/ (data/processed -> remonte de 2 crans)
_project_root = os.path.dirname(os.path.dirname(OUTPUT_DIR))
RESULTS_DIR = os.path.join(_project_root, "results")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("\nDossier de sortie (CSV)   :", OUTPUT_DIR)
print("Dossier de sortie (figures):", RESULTS_DIR)

## 3. Chargement des Données

On charge les données brutes/nettoyées ainsi que les deux sorties de M3 (`customer_segments.csv`)
et de M4 (`segment_recommendations.csv`, `electronic_recommendations.csv`). Si M4 n'a pas encore
été exécuté, on retombe sur un profilage minimal calculé à la volée pour ne pas bloquer M7.

In [ ]:
def load_all_datasets():
    customers = pd.read_csv(CUSTOMERS_PATH)
    sales = pd.read_csv(SALES_PATH)
    products = pd.read_csv(PRODUCTS_PATH) if PRODUCTS_PATH else pd.DataFrame()
    marketing = pd.read_csv(MARKETING_PATH)

    # --- Segments issus de M3 ---
    segments = pd.read_csv(SEGMENTS_PATH) if SEGMENTS_PATH else pd.DataFrame()

    # --- Recommandations issues de M4 ---
    segment_reco = (
        pd.read_csv(SEGMENT_RECO_PATH) if SEGMENT_RECO_PATH else pd.DataFrame()
    )
    electronic_reco = (
        pd.read_csv(ELECTRONIC_RECO_PATH) if ELECTRONIC_RECO_PATH else pd.DataFrame()
    )

    for df in [customers, sales, products, marketing, segments, segment_reco, electronic_reco]:
        if not df.empty:
            df.columns = [c.strip().lower() for c in df.columns]

    # --- Churn / CLV (M6) ---
    if CHURN_PATH and os.path.exists(CHURN_PATH):
        churn_clv = pd.read_csv(CHURN_PATH)
        churn_clv.columns = [c.strip().lower() for c in churn_clv.columns]
    else:
        # Fallback : reproduit une logique simplifiée de churn/CLV si M6
        # n'a pas encore été exécuté, pour que M7 reste autonome.
        sales["date"] = pd.to_datetime(sales["date"], errors="coerce")
        sales["amount"] = sales["quantity"] * sales["sale_price"]
        ref_date = sales["date"].max()
        cutoff = ref_date - pd.DateOffset(months=3)

        hist = sales[sales["date"] < cutoff]
        future = sales[sales["date"] >= cutoff]
        fut_buyers = set(future["customer_id"].unique())

        grp = hist.groupby("customer_id")
        freq = grp.size()
        spent = grp["amount"].sum()
        basket = grp["amount"].mean()
        last_p = grp["date"].max()
        rec = (cutoff - last_p).dt.days

        online_sales = hist[hist["channel"].str.lower() == "online"].groupby("customer_id").size()
        onl = (online_sales / freq).reindex(freq.index, fill_value=0)

        churn_clv = pd.DataFrame({
            "customer_id": freq.index,
            "frequency": freq.values,
            "total_spent_history": spent.values,
            "avg_basket": basket.values,
            "recency_days": rec.values,
            "online_ratio": onl.values,
        })
        churn_clv = customers.merge(churn_clv, on="customer_id", how="left").fillna(0)
        churn_clv["churn"] = churn_clv["customer_id"].apply(lambda cid: 0 if cid in fut_buyers else 1)
        churn_clv["churn_probability"] = np.where(churn_clv["recency_days"] > 90, 0.75, 0.20)
        churn_clv["clv_predictive_total"] = np.round(churn_clv["total_spent_history"] * 1.6 + 100, 2)

        def base_prio(r):
            if r["clv_predictive_total"] > 1500 and r["churn_probability"] >= 0.5:
                return "P1 - Retention VIP Immediat"
            elif r["clv_predictive_total"] > 1500:
                return "P2 - Ambassadeur / Fidelisation"
            elif r["churn_probability"] >= 0.5:
                return "P4 - Attrition Toleree"
            return "P3 - Developpement Panier"

        churn_clv["marketing_priority"] = churn_clv.apply(base_prio, axis=1)

    return customers, sales, products, marketing, segments, segment_reco, electronic_reco, churn_clv


(customers_df, sales_df, products_df, marketing_df,
 segments_df, segment_reco_df, electronic_reco_df, base_churn_clv) = load_all_datasets()

print(f"Data charges : {len(customers_df)} clients, {len(sales_df)} ventes, {len(marketing_df)} campagnes.")
print(f"Segments (M3)               : {len(segments_df)} lignes")
print(f"Recommandations segment (M4): {len(segment_reco_df)} lignes")
print(f"Recommandations electronic (M4): {len(electronic_reco_df)} lignes")

## 4. Analyse des performances des canaux (`marketing_clean.csv`)

In [ ]:
channel_stats = marketing_df.groupby("channel").agg({
    "campaign_id": "count",
    "budget": "sum",
    "impressions": "sum",
    "clicks": "sum",
    "conversions": "sum"
}).rename(columns={"campaign_id": "nb_campagnes"})

channel_stats["ctr_pct"] = np.round((channel_stats["clicks"] / channel_stats["impressions"]) * 100, 2)
channel_stats["taux_conv_pct"] = np.round((channel_stats["conversions"] / channel_stats["clicks"]) * 100, 2)
channel_stats["cpc_eur"] = np.round(channel_stats["budget"] / channel_stats["clicks"], 2)
channel_stats["cpa_eur"] = np.round(channel_stats["budget"] / channel_stats["conversions"], 2)

channel_stats = channel_stats.sort_values(by="cpa_eur", ascending=True)
display(channel_stats)
channel_stats.to_csv(f"{OUTPUT_DIR}/m7_channel_performance.csv")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.barplot(data=channel_stats.reset_index(), x="channel", y="ctr_pct", ax=axes[0, 0])
axes[0, 0].set_title("CTR (%) par Canal")

sns.barplot(data=channel_stats.reset_index(), x="channel", y="taux_conv_pct", ax=axes[0, 1])
axes[0, 1].set_title("Taux de Conversion (%)")

sns.barplot(data=channel_stats.reset_index(), x="channel", y="cpa_eur", ax=axes[1, 0])
axes[1, 0].set_title("CPA (Coût d'Acquisition en €)")

sns.barplot(data=channel_stats.reset_index(), x="channel", y="budget", ax=axes[1, 1])
axes[1, 1].set_title("Budget Total Consomme (€)")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/m7_channel_kpis.png")
plt.show()

## 5. Fusion Segments (M3) + Recommandations (M4) + Churn x CLV (M6)

Étape clé de compatibilité : on rattache à chaque client son `segment` (sortie M3), puis
on enrichit directement avec le texte de stratégie déjà produit par M4
(`segment_recommendations.csv`, `electronic_recommendations.csv`) plutôt que d'inventer
un nouveau texte, pour que M7 reste cohérent avec les conclusions de M4.

In [ ]:
# --- Rattacher le segment (M3) ---
if not segments_df.empty and "customer_id" in segments_df.columns:
    m7_merged = base_churn_clv.merge(segments_df[["customer_id", "segment"]], on="customer_id", how="left")
    m7_merged["segment"] = m7_merged["segment"].fillna(0).astype(int)
else:
    m7_merged = base_churn_clv.copy()
    m7_merged["segment"] = 0

# --- Rattacher les recommandations de segment (M4) ---
if not segment_reco_df.empty and "segment" in segment_reco_df.columns:
    reco_cols = [c for c in ["segment", "target", "marketing_strategy", "electronic_accessories_strategy"]
                 if c in segment_reco_df.columns]
    m7_merged = m7_merged.merge(segment_reco_df[reco_cols], on="segment", how="left")
else:
    m7_merged["target"] = "Non défini (M4 non exécuté)"
    m7_merged["marketing_strategy"] = "Stratégie générique"
    m7_merged["electronic_accessories_strategy"] = "Offre standard"

# --- Rattacher l'affinité Electronic Accessories (M4) ---
if not electronic_reco_df.empty and "segment" in electronic_reco_df.columns:
    elec_cols = [c for c in ["segment", "electronic_affinity", "objective", "recommendation"]
                 if c in electronic_reco_df.columns]
    m7_merged = m7_merged.merge(
        electronic_reco_df[elec_cols].rename(columns={"recommendation": "electronic_recommendation"}),
        on="segment", how="left"
    )
else:
    m7_merged["electronic_affinity"] = "Non défini"
    m7_merged["electronic_recommendation"] = "Offre standard"

m7_merged.head()

In [ ]:
segment_cross = pd.crosstab(
    m7_merged["segment"],
    m7_merged["marketing_priority"],
    margins=True
)
display(segment_cross)

cross_plot = pd.crosstab(m7_merged["segment"], m7_merged["marketing_priority"])
cross_plot.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.title("Repartition des priorites par segment")
plt.xlabel("Segment")
plt.ylabel("Nombre de clients")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/m7_segments_churn_cross.png")
plt.show()

## 6. Moteur de ciblage individuel

Le canal et le budget restent pilotés par la priorité churn/CLV (M6), mais le **contenu
de l'offre** (`targeted_content`) combine désormais directement la stratégie marketing et
la stratégie "Electronic Accessories" définies dans M4 pour ce segment, au lieu d'un texte
recréé indépendamment.

In [ ]:
def assign_targeting(row):
    prio = row["marketing_priority"]
    online_ratio = row["online_ratio"]
    clv = row["clv_predictive_total"]
    m4_strategy = row.get("marketing_strategy", "Stratégie générique")
    m4_elec_strategy = row.get("electronic_accessories_strategy", "Offre standard")

    if "P1" in prio:
        channel = "Direct VIP (Telephone / SMS / Mail)"
        content = f"Remise immédiate 25% + Bon d'achat 30€ | {m4_strategy}"
        budget = np.round(min(clv * 0.05, 50.0), 2)
    elif "P2" in prio:
        channel = "Emailing VIP" if online_ratio > 0.5 else "Invitation In-Store"
        content = f"Accès Ventes Privées + Points fidélité x2 | {m4_strategy}"
        budget = np.round(min(clv * 0.03, 30.0), 2)
    elif "P3" in prio:
        channel = "Ads Ciblées / Newsletter" if online_ratio > 0.5 else "Promo boutique"
        content = f"{m4_elec_strategy}"
        budget = np.round(min(clv * 0.02, 15.0), 2)
    else:
        channel = "Email Automatisé"
        content = f"Code promo 10% de relance | {m4_elec_strategy}"
        budget = np.round(min(clv * 0.008, 5.0), 2)

    return pd.Series(
        [channel, content, budget],
        index=["targeted_channel", "targeted_content", "allocated_budget_eur"]
    )

res = m7_merged.apply(assign_targeting, axis=1)
df_final = pd.concat([m7_merged, res], axis=1)
df_final.to_csv(f"{OUTPUT_DIR}/m7_digital_marketing_plan.csv", index=False)
display(df_final[[
    "customer_id", "segment", "clv_predictive_total", "marketing_priority",
    "targeted_channel", "targeted_content", "allocated_budget_eur"
]].head())

## 7. Modélisation du Budget Global (Simulation 100 000 €)

In [ ]:
GLOBAL_BUDGET = 100000.0
raw_weights = df_final.groupby("marketing_priority")["allocated_budget_eur"].sum()
budget_normalized = (raw_weights / raw_weights.sum()) * GLOBAL_BUDGET

budget_df = pd.DataFrame({
    "nb_clients": df_final.groupby("marketing_priority").size(),
    "budget_total": np.round(budget_normalized, 2),
    "part_pct": np.round((budget_normalized / GLOBAL_BUDGET) * 100, 1),
    "budget_moyen_client": np.round(budget_normalized / df_final.groupby("marketing_priority").size(), 2)
})

display(budget_df)
budget_df.to_csv(f"{OUTPUT_DIR}/m7_budget_allocation.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
budget_df["budget_total"].plot(kind="pie", autopct="%1.1f%%", ax=axes[0])
axes[0].set_ylabel("")
axes[0].set_title("Répartition du Budget (100k €)")

budget_df["budget_moyen_client"].plot(kind="barh", ax=axes[1])
axes[1].set_title("Budget Moyen par Client (€)")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/m7_budget_distribution.png")
plt.show()

## 8. Plan Stratégique Opérationnel

### Roadmap d'exécution :
1. **Mois 1-2 (Phase 1) :** Activation des campagnes d'urgence sur le groupe P1 (Rétention VIP). Utilisation du canal direct (téléphone/SMS).
2. **Mois 3-4 (Phase 2) :** Lancement des ventes privées pour le groupe P2 et campagnes de cross-selling accessoires pour le groupe P3, en s'appuyant sur les stratégies définies en M4.
3. **Mois 5-6 (Phase 3) :** Automatisation des flux d'emailing de réactivation pour le groupe P4.

### Chaîne de compatibilité M3 → M4 → M7 :
- `M3` produit `customer_segments.csv` (customer_id, segment) dans `data/processed/`.
- `M4` produit `segment_recommendations.csv` et `electronic_recommendations.csv` dans `data/processed/`.
- `M7` consomme ces trois fichiers et écrit ses propres sorties **dans le même dossier
  que celui où il a trouvé les fichiers M3/M4** (variable `OUTPUT_DIR`, calculée
  automatiquement en section 2 — ce n'est plus un chemin relatif fixe) :
  - `m7_channel_performance.csv`
  - `m7_digital_marketing_plan.csv`
  - `m7_budget_allocation.csv`


In [ ]:
print("=== Récapitulatif des fichiers générés ===")
for fname in ["m7_channel_performance.csv", "m7_digital_marketing_plan.csv", "m7_budget_allocation.csv"]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    print(f"{fname:35s} -> {'OK' if os.path.exists(fpath) else 'ABSENT'} ({fpath})")